# 🛡️ Spam Email Classifier
### End-to-End ML Project: NLP · TF-IDF · Naive Bayes · SVM
---

## 1. Setup & Imports

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix

from src.preprocess import load_and_preprocess, get_word_frequencies
from src.utils import compute_metrics, plot_class_distribution, plot_word_frequency, plot_confusion_matrix, plot_accuracy_comparison

print('✅ All imports successful')

## 2. Load & Explore Dataset

In [ ]:
df = load_and_preprocess('../dataset/spam.csv')
print('\nShape:', df.shape)
df.head()

In [ ]:
print('Label distribution:')
print(df['label'].value_counts())
print('\nMissing values:', df.isnull().sum().sum())

# Average message length by class
df['text_length'] = df['text'].apply(len)
print('\nAvg message length:')
print(df.groupby('label')['text_length'].mean().round(1))

In [ ]:
fig = plot_class_distribution(df)
plt.show()

## 3. NLP Preprocessing Deep Dive

In [ ]:
# Show preprocessing effect on a spam example
spam_example = df[df['label']=='spam']['text'].iloc[0]
spam_clean   = df[df['label']=='spam']['clean_text'].iloc[0]
print('ORIGINAL :', spam_example)
print('\nCLEANED  :', spam_clean)

In [ ]:
ham_freq  = get_word_frequencies(df, 'ham',  top_n=15)
spam_freq = get_word_frequencies(df, 'spam', top_n=15)
fig = plot_word_frequency(ham_freq, spam_freq)
plt.show()

## 4. TF-IDF Feature Engineering

In [ ]:
X = df['clean_text']
y = df['label_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit TF-IDF on training data only (prevent data leakage)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True, min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Train TF-IDF matrix: {X_train_tfidf.shape}')
print(f'Test  TF-IDF matrix: {X_test_tfidf.shape}')
print(f'Vocabulary size    : {len(tfidf.vocabulary_)}')

## 5. Train & Evaluate Models

In [ ]:
tfidf_params = dict(max_features=5000, ngram_range=(1,2), sublinear_tf=True, min_df=2)

nb_pipeline  = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', MultinomialNB(alpha=0.1))])
svm_pipeline = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=2000, random_state=42)))])

nb_pipeline.fit(X_train, y_train)
svm_pipeline.fit(X_train, y_train)
print('✅ Both models trained')

In [ ]:
nb_pred  = nb_pipeline.predict(X_test)
svm_pred = svm_pipeline.predict(X_test)

nb_metrics  = compute_metrics(y_test, nb_pred,  'Naive Bayes')
svm_metrics = compute_metrics(y_test, svm_pred, 'SVM (LinearSVC)')

In [ ]:
fig = plot_confusion_matrix(y_test, nb_pred,  'Naive Bayes')
plt.show()
fig = plot_confusion_matrix(y_test, svm_pred, 'SVM (LinearSVC)')
plt.show()

In [ ]:
fig = plot_accuracy_comparison([nb_metrics, svm_metrics])
plt.show()

## 6. Cross-Validation

In [ ]:
for name, pipe in [('Naive Bayes', nb_pipeline), ('SVM', svm_pipeline)]:
    scores = cross_val_score(pipe, X, y, cv=5, scoring='f1')
    print(f'{name}: {scores.mean():.4f} ± {scores.std():.4f}')

## 7. Save Models

In [ ]:
import joblib, os
os.makedirs('../models', exist_ok=True)
joblib.dump(nb_pipeline,  '../models/naive_bayes.pkl')
joblib.dump(svm_pipeline, '../models/svm.pkl')
print('✅ Models saved to /models/')

## 8. Test Predictions

In [ ]:
from src.predict import SpamPredictor
predictor = SpamPredictor('svm')

test_emails = [
    'WINNER!! You have been selected for a FREE prize. Call now!',
    'Hey, are you coming to the meeting tomorrow at 3pm?',
    'Claim your £500 cash reward. Text WIN to 80888.',
    'The project report has been uploaded to the shared drive.',
]

for email in test_emails:
    res = predictor.predict(email)
    icon = '🚫' if res['label']=='Spam' else '✅'
    print(f"{icon} [{res['label']:4s}] {res['confidence']*100:.1f}% | {email[:60]}")